# UniVTG Fixed vs Individual — Top-50 후보 생성 (Preview OFF v1)

이 노트북은 **평가용이 아니라 UniVTG 추론·후보 저장용**입니다.

- 각 영상에 대해 Fixed / Individual Query를 각각 실행합니다.
- Query별 점수를 비교해 `combined_score`를 계산합니다.
- 영상·모드별 **Top-50 후보 장면**을 `candidate_segments_top50.csv`에 저장합니다.
- 최종 Top-4는 `selected_segments.csv`에 저장하며, **Preview MP4는 생성하지 않습니다.**
- 게임 장면 길이는 **5초**, 영화·팟캐스트는 **15초**입니다.
- `train`, `val_a`, `val_b`, `test_a`, `test_b`를 Metadata의 `split` 컬럼에서 우선 읽고, 없으면 파일명에서 자동 추론합니다.
- Metadata와 Query CSV가 여러 파일로 늘어나도 자동 검색·결합합니다.
- 같은 `video_id`가 여러 Query 버전에 중복되면 최신 파일 우선 또는 명시적 파일 선택으로 처리합니다.

> 실행 속도 최적화: `CREATE_PREVIEW=False`로 고정되어 Top-4 영상 인코딩·병합·Drive 저장을 생략합니다.


## 1. 환경·입력 자동 탐색·출력 경로

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

from pathlib import Path
import torch


# ============================================================
# 1. GPU 확인
# ============================================================
if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU가 없습니다. 런타임 → 런타임 유형 변경 → GPU를 선택하세요.'
    )


# ============================================================
# 2. 프로젝트 경로
# ============================================================
PROJECT_ROOT = Path(
    '/content/drive/MyDrive/'
    '0_ASAC_11기_DL_3조'
)

UNIVTG_ROOT = PROJECT_ROOT / 'code' / 'UniVTG'
CSV_ROOT = UNIVTG_ROOT / 'csv'
ACTIVE_CSV_ROOT = CSV_ROOT / 'active'

DATA_ROOT = PROJECT_ROOT / '유튜브데이터'
VIDEO_ROOT = DATA_ROOT / '영상'

# 사용자가 지정한 active 폴더 기준 Query CSV 사용
ACTIVE_QUERY_DIR = ACTIVE_CSV_ROOT

# 영상은 기본 YouTube 영상 폴더를 먼저 찾고, 혹시 active 쪽에 영상이 같이 있어도 찾도록 추가합니다.
VIDEO_SEARCH_ROOTS = tuple(dict.fromkeys([
    VIDEO_ROOT,
    ACTIVE_CSV_ROOT,
    UNIVTG_ROOT / 'videos',
    UNIVTG_ROOT / 'video',
]))


# ============================================================
# 3. Final-920 실행 설정
# ============================================================
EXPECTED_TARGET_VIDEO_COUNT = 920
FINAL_TARGET_VIDEO_COUNT = 920
STRICT_TARGET_VIDEO_COUNT = False  # True면 실행 가능 영상 수가 920이 아닐 때 중단

# train_intro_cut.csv는 active 폴더에 있어도 이번 Top-50 생성에서는 사용하지 않습니다.
IGNORE_INTRO_CUT_FOR_TOP50 = True


# ============================================================
# 4. Active Query CSV 자동 검색
# ============================================================
QUERY_FILE_PATHS = tuple(sorted(
    path
    for path in ACTIVE_QUERY_DIR.glob('*query_dataset*.csv')
    if path.is_file()
))

if not QUERY_FILE_PATHS:
    raise FileNotFoundError(
        'active 폴더에서 Query Dataset CSV를 찾지 못했습니다.\n'
        f'검색 위치: {ACTIVE_QUERY_DIR}\n'
        "파일명에 'query_dataset'이 포함되어야 합니다."
    )

QUERY_CONFLICT_POLICY = 'deduplicate_keep_last'


# ============================================================
# 5. 실행 Split
# ============================================================
ACTIVE_SPLITS = None
SUPPORTED_SPLITS = {
    'train',
    'val',
    'test',
    'val_a',
    'val_b',
    'test_a',
    'test_b',
}


# ============================================================
# 6. 결과·작업 폴더
# ============================================================
RESULT_ROOT = UNIVTG_ROOT / 'result_top50_final920'
REPO_DIR = Path('/content/UniVTG')
WORK_ROOT = Path('/content/univtg_work_top50_final920')


# ============================================================
# 7. 출력 옵션
# ============================================================
# 평가용 CSV만 생성하고 Preview MP4 인코딩은 생략합니다.
CREATE_PREVIEW = False
SAVE_ALL_CLIP_SCORES = False


# ============================================================
# 8. 경로 검증
# ============================================================
required_paths = {
    '프로젝트 루트': PROJECT_ROOT,
    'UniVTG 루트': UNIVTG_ROOT,
    'CSV 폴더': CSV_ROOT,
    'Active CSV 폴더': ACTIVE_CSV_ROOT,
}
for label, path in required_paths.items():
    if not path.exists():
        raise FileNotFoundError(
            f'{label}가 없습니다:\n{path}'
        )

existing_video_roots = [path for path in VIDEO_SEARCH_ROOTS if path.exists()]
if not existing_video_roots:
    raise FileNotFoundError(
        '영상 검색 루트가 하나도 없습니다. 아래 후보 경로를 확인하세요:\n'
        + '\n'.join(f' - {path}' for path in VIDEO_SEARCH_ROOTS)
    )

RESULT_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print('=' * 80)
print('Final-920 경로 및 입력 설정 완료')
print('=' * 80)
print('GPU:', torch.cuda.get_device_name(0))
print('CSV 루트:', CSV_ROOT)
print('Active Query CSV 루트:', ACTIVE_QUERY_DIR)
print('결과 폴더:', RESULT_ROOT)
print('작업 폴더:', WORK_ROOT)
print('목표 영상 수:', FINAL_TARGET_VIDEO_COUNT)
print('Strict 목표 개수 검사:', STRICT_TARGET_VIDEO_COUNT)
print('인트로 컷 사용 여부:', '사용 안 함' if IGNORE_INTRO_CUT_FOR_TOP50 else '사용')
print('활성 Split:', 'ALL' if ACTIVE_SPLITS is None else sorted(ACTIVE_SPLITS))
print('Preview 생성:', CREATE_PREVIEW)
print('전체 Clip 점수 저장:', SAVE_ALL_CLIP_SCORES)
print('\n[영상 검색 루트]')
for path in existing_video_roots:
    print(' -', path)
print('\n[사용할 Query CSV]')
for path in QUERY_FILE_PATHS:
    print(' -', path)


## 2. UniVTG 저장소와 패키지

In [ ]:
import subprocess
import sys

REPO_COMMIT = '32659ac7aeba21742a63274f30eba785fc57e247'

if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/showlab/UniVTG.git', str(REPO_DIR)],
        check=True,
    )

subprocess.run(
    ['git', '-C', str(REPO_DIR), 'checkout', REPO_COMMIT],
    check=True,
)

PACKAGES = [
    'gdown',
    'decord==0.6.0',
    'ftfy==6.1.1',
    'regex',
    'ffmpeg-python==0.2.0',
    'easydict',
    'einops',
    'timm',
    'tensorboard',
    'tabulate',
    'tqdm',
]

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', *PACKAGES],
    check=True,
)

print('UniVTG 저장소:', REPO_DIR)
print('고정 commit:', REPO_COMMIT)
print('추가 패키지 설치 완료')

## 3. 공식 체크포인트

In [ ]:
import shutil

CHECKPOINT_DIR = REPO_DIR / 'results' / 'omni'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT = CHECKPOINT_DIR / 'model_best.ckpt'
OPT_JSON = CHECKPOINT_DIR / 'opt.json'

if not (CHECKPOINT.exists() and OPT_JSON.exists()):
    download_dir = Path('/content/univtg_checkpoint_download')
    shutil.rmtree(download_dir, ignore_errors=True)
    download_dir.mkdir(parents=True)

    subprocess.run(
        [
            'gdown', '--folder',
            'https://drive.google.com/drive/folders/'
            '1l6RyjGuqkzfZryCC6xwTZsvjWaIMVxIO?usp=sharing',
            '-O', str(download_dir),
        ],
        check=True,
    )

    checkpoint_matches = list(download_dir.rglob('model_best.ckpt'))
    option_matches = list(download_dir.rglob('opt.json'))

    if not checkpoint_matches or not option_matches:
        raise FileNotFoundError(
            '공식 체크포인트에서 model_best.ckpt 또는 opt.json을 찾지 못했습니다.'
        )

    shutil.copy2(checkpoint_matches[0], CHECKPOINT)
    shutil.copy2(option_matches[0], OPT_JSON)

print('체크포인트:', CHECKPOINT)
print('opt.json:', OPT_JSON)

## 4. CLIP과 UniVTG 모델 로드

In [ ]:
import argparse
import os
import numpy as np
import pandas as pd

os.chdir(REPO_DIR)

for old_name, replacement in {
    'float': float,
    'int': int,
    'complex': complex,
}.items():
    if not hasattr(np, old_name):
        setattr(np, old_name, replacement)

from run_on_video import clip, vid2clip, txt2clip
from main.config import TestOptions, setup_model
from utils.basic_utils import l2_normalize_np_array

DEVICE = torch.device('cuda:0')
CLIP_MODEL_NAME = 'ViT-B/32'

if 'clip_model' not in globals():
    print('CLIP 로드 중...')
    clip_model, _ = clip.load(
        CLIP_MODEL_NAME,
        device=DEVICE,
        jit=False,
    )
    clip_model.eval()

if not hasattr(torch, '_univtg_original_load'):
    torch._univtg_original_load = torch.load


def compatible_torch_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return torch._univtg_original_load(*args, **kwargs)


torch.load = compatible_torch_load

if 'univtg_model' not in globals():
    old_argv = sys.argv[:]
    sys.argv = ['univtg_colab']

    try:
        args = argparse.Namespace(resume=str(CHECKPOINT), gpu_id=0)
        option = TestOptions().parse(args)

        if isinstance(option.lr_warmup, (int, float)):
            option.lr_warmup = [int(option.lr_warmup)]

        univtg_model, _, _, _ = setup_model(option)
        univtg_model.eval()
    finally:
        sys.argv = old_argv

print('모델 로드 완료:', torch.cuda.get_device_name(0))

## 5. Query·Metadata·본문 시간 CSV 자동 검색·검증·결합

In [ ]:

import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Video, display


CODE_VERSION = 'final920_active_query_only_2026-07-08_v1'
DEFAULT_METADATA_SPLIT = 'train'
BODY_END_FALLBACK_SEC = 1_000_000_000.0

print(f'[CODE VERSION] {CODE_VERSION}')


def normalize_text(value) -> str:
    if pd.isna(value):
        return ''
    return unicodedata.normalize('NFC', str(value).strip())


def normalize_column_name(value) -> str:
    return normalize_text(value).lower().replace(' ', '_')


def canonical_category(value) -> str:
    key = (
        normalize_text(value)
        .lower()
        .replace('-', '_')
        .replace(' ', '_')
    )

    aliases = {
        'game': 'game',
        'gaming': 'game',
        'games': 'game',
        '게임': 'game',

        'movie': 'movie',
        'movie_review': 'movie',
        'moviereview': 'movie',
        'drama': 'movie',
        'film': 'movie',
        '영화': 'movie',

        'podcast': 'podcast',
        'podcasts': 'podcast',
        'talk': 'podcast',
        '팟캐스트': 'podcast',
    }

    if key not in aliases:
        raise ValueError(
            f'지원하지 않는 category: {value!r}'
        )

    return aliases[key]


def canonical_split(value) -> str:
    key = (
        normalize_text(value)
        .lower()
        .replace('-', '_')
        .replace(' ', '_')
    )

    aliases = {
        'train': 'train',
        'training': 'train',

        'val': 'val',
        'valid': 'val',
        'validation': 'val',
        'dev': 'val',

        'test': 'test',
        'testing': 'test',

        # legacy names also supported
        'val_a': 'val_a',
        'vala': 'val_a',
        'validation_a': 'val_a',
        'val_b': 'val_b',
        'valb': 'val_b',
        'validation_b': 'val_b',
        'test_a': 'test_a',
        'testa': 'test_a',
        'test_b': 'test_b',
        'testb': 'test_b',
    }

    if key not in aliases:
        raise ValueError(
            f'지원하지 않는 split: {value!r}'
        )

    return aliases[key]


def infer_split_from_text(value) -> str:
    text = (
        normalize_text(value)
        .lower()
        .replace('\\', '_')
        .replace('/', '_')
        .replace('-', '_')
        .replace(' ', '_')
    )

    patterns = [
        ('test_a', r'(^|_)test_?a(_|$)'),
        ('test_b', r'(^|_)test_?b(_|$)'),
        ('val_a', r'(^|_)val(?:idation)?_?a(_|$)'),
        ('val_b', r'(^|_)val(?:idation)?_?b(_|$)'),
        ('test', r'(^|_)test(?:ing)?(_|$)'),
        ('val', r'(^|_)val(?:id|idation)?(_|$)'),
        ('train', r'(^|_)train(?:ing)?(_|$)'),
    ]

    for split, pattern in patterns:
        if re.search(pattern, text):
            return split

    return ''


def infer_category_from_text(value) -> str:
    text = (
        normalize_text(value)
        .lower()
        .replace('\\', '_')
        .replace('/', '_')
        .replace('-', '_')
        .replace(' ', '_')
    )

    patterns = [
        ('movie', ('movie_review', 'moviereview', 'movie', 'drama', 'film', '영화')),
        ('podcast', ('podcast', 'talk', '팟캐스트')),
        ('game', ('gaming', 'game', 'games', '게임')),
    ]

    for category, tokens in patterns:
        if any(token in text for token in tokens):
            return category

    return ''


def query_column_number(column: str) -> int:
    match = re.search(r'_(\d+)$', column)
    return int(match.group(1)) if match else 9999


def read_csv(path: Path, *, uppercase: bool = False) -> pd.DataFrame:
    frame = pd.read_csv(path, encoding='utf-8-sig', dtype=str)
    if uppercase:
        frame.columns = [normalize_text(column).upper() for column in frame.columns]
    else:
        frame.columns = [normalize_column_name(column) for column in frame.columns]
    frame['_source_csv'] = str(path)
    return frame


def require_columns(frame: pd.DataFrame, required: list[str], label: str) -> None:
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(
            f'{label} 누락 컬럼: {missing}\n'
            f'현재 컬럼: {frame.columns.tolist()}'
        )


def coalesce_column(frame: pd.DataFrame, target: str, aliases: list[str]) -> None:
    if target not in frame.columns:
        frame[target] = ''
    result = frame[target].map(normalize_text)
    for alias in aliases:
        if alias in frame.columns:
            candidate = frame[alias].map(normalize_text)
            result = result.mask(result.eq(''), candidate)
    frame[target] = result


def validate_and_deduplicate(frame: pd.DataFrame, *, label: str, compare_columns: list[str]) -> pd.DataFrame:
    if frame.empty:
        return frame
    duplicated = frame['video_id'].duplicated(keep=False)
    duplicated_frame = frame.loc[duplicated]
    for video_id, group in duplicated_frame.groupby('video_id'):
        conflicts = {}
        for column in compare_columns:
            if column not in group.columns:
                continue
            values = [normalize_text(value) for value in group[column] if normalize_text(value)]
            unique = sorted(set(values))
            if len(unique) > 1:
                conflicts[column] = unique
        if conflicts:
            raise ValueError(
                f'{label}에서 video_id={video_id}의 값이 충돌합니다: {conflicts}'
            )
    return frame.drop_duplicates(subset='video_id', keep='first').reset_index(drop=True)


def query_file_sort_key(path: Path):
    name = normalize_text(path.name).lower()
    order = 99
    if 'train' in name:
        order = 0
    elif 'val' in name:
        order = 1
    elif 'test' in name:
        order = 2
    return (order, name)


# ============================================================
# 5-1. Active Query CSV
# ============================================================

query_files = sorted(list(QUERY_FILE_PATHS), key=query_file_sort_key)

print('\n' + '=' * 80)
print('실제 사용 Query 파일')
print('=' * 80)
for path in query_files:
    print(' -', path)

missing_query_files = [path for path in query_files if not path.is_file()]
if missing_query_files:
    raise FileNotFoundError(
        '지정한 Query CSV가 없습니다:\n'
        + '\n'.join(f' - {path}' for path in missing_query_files)
    )

query_frames = []
for path in query_files:
    frame = read_csv(path)

    coalesce_column(frame, 'video_id', ['id', 'youtube_id', 'videoid', 'video_key'])
    coalesce_column(frame, 'title', ['video_title', 'name'])
    coalesce_column(frame, 'category', ['genre', 'class'])
    coalesce_column(frame, 'creator_name', ['channel', 'channel_name', 'creator', 'youtuber'])
    coalesce_column(frame, 'duration_sec', ['duration', 'video_duration', 'length_sec'])
    coalesce_column(frame, 'local_filename', ['filename', 'file_name', 'video_filename', 'video_file'])
    coalesce_column(frame, 'split', ['dataset_split', 'role', 'set'])

    inferred_split = infer_split_from_text(path.name)
    frame['_source_split'] = inferred_split
    frame['split'] = frame['split'].mask(frame['split'].eq(''), inferred_split)
    frame['split'] = frame['split'].mask(frame['split'].eq(''), DEFAULT_METADATA_SPLIT)

    query_frames.append(frame)

query_df = pd.concat(query_frames, ignore_index=True, sort=False)

FIXED_QUERY_COLUMNS = sorted(
    [column for column in query_df.columns if re.fullmatch(r'fixed_query_\d+', column)],
    key=query_column_number,
)
INDIV_QUERY_COLUMNS = sorted(
    [column for column in query_df.columns if re.fullmatch(r'indiv_query_\d+', column)],
    key=query_column_number,
)

require_columns(query_df, ['video_id', 'title', 'category'], 'Query CSV')

if not FIXED_QUERY_COLUMNS:
    raise ValueError('Fixed Query 컬럼을 찾지 못했습니다.')
if not INDIV_QUERY_COLUMNS:
    raise ValueError('Individual Query 컬럼을 찾지 못했습니다.')
if len(FIXED_QUERY_COLUMNS) != len(INDIV_QUERY_COLUMNS):
    raise ValueError(
        'Fixed/Individual Query 개수가 다릅니다: '
        f'{len(FIXED_QUERY_COLUMNS)} / {len(INDIV_QUERY_COLUMNS)}'
    )

QUERY_COUNT = len(FIXED_QUERY_COLUMNS)
QUERY_TEXT_COLUMNS = [*FIXED_QUERY_COLUMNS, *INDIV_QUERY_COLUMNS]

for column in [
    'video_id', 'title', 'category', 'creator_name', 'duration_sec',
    'local_filename', 'split', *QUERY_TEXT_COLUMNS,
]:
    query_df[column] = query_df[column].map(normalize_text)

if query_df['video_id'].eq('').any():
    display(query_df.loc[query_df['video_id'].eq('')])
    raise ValueError('Query CSV에 빈 video_id가 있습니다.')

query_df['_query_row_order'] = np.arange(len(query_df))
duplicate_query_df = query_df.loc[
    query_df['video_id'].duplicated(keep=False),
    ['video_id', 'title', 'category', 'split', '_source_csv', '_query_row_order', *QUERY_TEXT_COLUMNS],
].copy()

if not duplicate_query_df.empty:
    print('\n[경고] Query CSV 중복 video_id 발견:', duplicate_query_df['video_id'].nunique())
    print('처리 방식: 같은 video_id 중 CSV에서 가장 마지막 행 사용')
    display(
        duplicate_query_df.groupby('video_id').size().rename('duplicate_row_count').reset_index().head(30)
    )

query_df = (
    query_df
    .sort_values(['video_id', '_query_row_order'], kind='mergesort')
    .drop_duplicates(subset=['video_id'], keep='last')
    .drop(columns=['_query_row_order'])
    .reset_index(drop=True)
)

empty_query_mask = query_df[QUERY_TEXT_COLUMNS].eq('').any(axis=1)
if empty_query_mask.any():
    display(query_df.loc[empty_query_mask, ['video_id', 'title', 'category', 'split', '_source_csv']])
    raise ValueError('비어 있는 Fixed/Individual Query가 있습니다.')

query_df['query_category'] = query_df['category'].map(canonical_category)
query_df['split'] = query_df['split'].map(canonical_split)

if ACTIVE_SPLITS is not None:
    active = {canonical_split(value) for value in ACTIVE_SPLITS}
    query_df = query_df[query_df['split'].isin(active)].copy()

if query_df.empty:
    raise ValueError('ACTIVE_SPLITS 적용 후 실행 가능한 Query가 없습니다.')

# ============================================================
# 5-2. Metadata / Intro Cut 미사용 Dataset 생성
# ============================================================
metadata_files = []
timing_files = []

metadata_df = query_df.copy()
metadata_df['metadata_category'] = metadata_df['query_category']
metadata_df['duration_sec'] = pd.to_numeric(metadata_df['duration_sec'], errors='coerce')
metadata_df['creator_name'] = metadata_df['creator_name'].mask(metadata_df['creator_name'].eq(''), 'unknown_creator')

# 이번 Final-920 Top-50 생성에서는 intro cut을 쓰지 않고 원본 영상 전체 기준으로 후보를 생성합니다.
metadata_df['intro_end_sec'] = np.nan
metadata_df['branding_end_sec'] = 0.0
metadata_df['video_end_sec'] = metadata_df['duration_sec']
metadata_df['video_end_sec'] = metadata_df['video_end_sec'].fillna(BODY_END_FALLBACK_SEC)


def fallback_stem(row):
    local_name = normalize_text(row.get('local_filename', ''))
    if local_name:
        return Path(local_name).stem
    return normalize_text(row['video_id'])

metadata_df['source_video_stem'] = metadata_df.apply(fallback_stem, axis=1)

# 최종 Dataset 생성
dataset = metadata_df.copy()
dataset['category'] = dataset['metadata_category']
dataset['title'] = dataset['title'].map(normalize_text)
dataset['body_start_sec'] = 0.0
dataset['body_end_sec'] = pd.to_numeric(dataset['video_end_sec'], errors='coerce').fillna(BODY_END_FALLBACK_SEC)
dataset['body_duration_sec'] = dataset['body_end_sec'] - dataset['body_start_sec']
dataset['_video_key'] = dataset['video_id'].map(normalize_text)

invalid_body = dataset[
    dataset[['body_start_sec', 'body_end_sec']].isna().any(axis=1)
    | (dataset['body_start_sec'] < 0)
    | (dataset['body_end_sec'] <= dataset['body_start_sec'])
]
if not invalid_body.empty:
    display(invalid_body[['video_id', 'split', 'body_start_sec', 'body_end_sec', 'duration_sec']])
    raise ValueError('유효하지 않은 본문 시작/종료 시간이 있습니다.')

if dataset['video_id'].duplicated().any():
    duplicated_ids = dataset.loc[dataset['video_id'].duplicated(keep=False), 'video_id'].tolist()
    raise ValueError('최종 Dataset에 중복 video_id가 있습니다: ' + str(duplicated_ids[:20]))

if len(dataset) != len(dataset['video_id'].unique()):
    raise AssertionError('Dataset 행 수와 고유 video_id 수가 다릅니다.')

# 실행 전 입력 보고서 저장
input_report_path = RESULT_ROOT / 'active_query_dataset_report.csv'
input_report = pd.DataFrame([{
    'query_file_count': len(query_files),
    'query_video_count': len(query_df),
    'dataset_video_count': len(dataset),
    'expected_target_video_count': EXPECTED_TARGET_VIDEO_COUNT,
    'final_target_video_count': FINAL_TARGET_VIDEO_COUNT,
    'ignore_intro_cut_for_top50': IGNORE_INTRO_CUT_FOR_TOP50,
}])
input_report.to_csv(input_report_path, index=False, encoding='utf-8-sig')

print('\n' + '=' * 80)
print('Active Query CSV 결합 완료')
print('=' * 80)
print('Query 파일:', len(query_files))
for path in query_files:
    print(' -', path)
print('\n최종 Query 영상:', len(dataset))
print('고유 video_id:', dataset['video_id'].nunique())
print('Fixed Query 수:', len(FIXED_QUERY_COLUMNS))
print('Individual Query 수:', len(INDIV_QUERY_COLUMNS))
print('Intro cut 사용:', '아니오')
print('입력 보고서:', input_report_path)

print('\nSplit별 영상 수')
display(dataset.groupby('split').size().rename('video_count').reset_index())
print('Category별 영상 수')
display(dataset.groupby('category').size().rename('video_count').reset_index())

display(dataset[[
    'video_id', 'split', 'creator_name', 'title', 'category',
    'body_start_sec', 'body_end_sec', 'source_video_stem', '_source_csv'
]].head(10))


## 6. 전체 Split 영상 파일 자동 연결

In [ ]:
VIDEO_EXTENSIONS = {'.mp4', '.mov', '.mkv', '.webm', '.avi', '.m4v'}

existing_video_roots = [path for path in VIDEO_SEARCH_ROOTS if path.exists()]
video_file_map = {}
for root in existing_video_roots:
    for path in root.rglob('*'):
        if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS:
            video_file_map[str(path.resolve())] = path
video_files = sorted(video_file_map.values())


def normalized_stem(path: Path) -> str:
    return normalize_text(path.stem)


stem_map: dict[str, list[Path]] = {}
for path in video_files:
    stem_map.setdefault(normalized_stem(path), []).append(path)


def path_preference_score(path: Path, row: pd.Series) -> int:
    text = normalize_text(str(path)).lower().replace('-', '_').replace(' ', '_')
    score = 0
    split = normalize_text(row['split'])
    category = canonical_category(row['category'])
    creator = normalize_text(row['creator_name']).lower()

    split_tokens = {
        'train': ('train',),
        'val': ('val', 'valid', 'validation'),
        'test': ('test', 'testing'),
        'val_a': ('val_a', 'vala', 'validation_a'),
        'val_b': ('val_b', 'valb', 'validation_b'),
        'test_a': ('test_a', 'testa'),
        'test_b': ('test_b', 'testb'),
    }[split]
    category_tokens = {
        'game': ('game', 'gaming', '게임'),
        'movie': ('movie', 'movie_review', '영화'),
        'podcast': ('podcast', '팟캐스트'),
    }[category]

    if any(token in text for token in split_tokens):
        score += 4
    if any(token in text for token in category_tokens):
        score += 2
    if creator and creator in text:
        score += 1
    return score


def choose_unique_match(matches: list[Path], row: pd.Series):
    if not matches:
        return None, 'not_found'
    if len(matches) == 1:
        return str(matches[0]), 'unique_match'

    scored = [(path_preference_score(path, row), path) for path in matches]
    best_score = max(score for score, _ in scored)
    best = [path for score, path in scored if score == best_score]
    if len(best) == 1:
        return str(best[0]), f'preferred_match_score_{best_score}'
    return None, f'ambiguous_{len(matches)}_matches'


def resolve_video(row: pd.Series) -> tuple[str | None, str]:
    # 1순위: Timing/Metadata의 긴 stem과 정확히 일치
    exact = stem_map.get(normalize_text(row['source_video_stem']), [])
    path, reason = choose_unique_match(exact, row)
    if path is not None:
        return path, 'exact_source_stem'
    if exact:
        return None, reason

    # 2순위: 파일명에 YouTube video_id 포함
    video_id = normalize_text(row['video_id'])
    matches = [
        path for path in video_files
        if video_id in normalize_text(path.stem)
    ]
    return choose_unique_match(matches, row)


resolved = dataset.apply(resolve_video, axis=1)
dataset['video_path'] = [item[0] for item in resolved]
dataset['video_match'] = [item[1] for item in resolved]

unresolved = dataset['video_path'].isna()
if unresolved.any():
    print('영상 경로를 찾지 못했거나 중복 매칭된 행:')
    display(dataset.loc[
        unresolved,
        ['video_id', 'split', 'creator_name', 'title', 'source_video_stem', 'video_match'],
    ])

runnable = dataset.loc[~unresolved].reset_index(drop=True)

print('영상 검색 루트:')
for root in existing_video_roots:
    print(' -', root)
print('전체 영상 파일:', len(video_files))
print('CSV 실험 대상:', len(dataset))
print('실행 가능한 영상:', len(runnable))
print('실행 가능 Split 분포:')
display(runnable.groupby('split').size().rename('video_count').reset_index())

if runnable.empty:
    raise RuntimeError('실행 가능한 영상이 없습니다.')

display(runnable[[
    'video_id', 'split', 'creator_name', 'category', 'video_path', 'video_match'
]].head(10))

## 7. Top-50 후보·Top-4 Preview 실험 규칙

In [ ]:
import hashlib
import json

CATEGORY_SETTINGS = {
    'game': {
        'analysis_clip_sec': 2.0,
        'before_sec': 2.5,
        'after_sec': 2.5,
        'candidate_top_n': 50,
        'final_top_k': 4,
        'final_min_gap_sec': 2.0,
    },
    'movie': {
        'analysis_clip_sec': 2.0,
        'before_sec': 5.0,
        'after_sec': 10.0,
        'candidate_top_n': 50,
        'final_top_k': 4,
        'final_min_gap_sec': 5.0,
    },
    'podcast': {
        'analysis_clip_sec': 2.0,
        'before_sec': 5.0,
        'after_sec': 10.0,
        'candidate_top_n': 50,
        'final_top_k': 4,
        'final_min_gap_sec': 5.0,
    },
}

SALIENCY_WEIGHT = 0.75
FOREGROUND_WEIGHT = 0.25
SCORE_AGGREGATION = 'max'  # max / mean / top2_mean


def safe_name(value) -> str:
    text = normalize_text(value)
    return ''.join(
        character if character.isalnum() or character in '-_' else '_'
        for character in text
    )


def build_config(row: pd.Series, mode: str) -> dict:
    if mode not in {'fixed', 'individual'}:
        raise ValueError('mode는 fixed 또는 individual이어야 합니다.')

    category = canonical_category(row['category'])
    query_columns = FIXED_QUERY_COLUMNS if mode == 'fixed' else INDIV_QUERY_COLUMNS
    queries = [
        normalize_text(row[column])
        for column in query_columns
        if normalize_text(row[column])
    ]
    if len(queries) != QUERY_COUNT:
        raise ValueError(
            f"{row['video_id']} / {mode}: Query가 {QUERY_COUNT}개여야 하지만 "
            f'{len(queries)}개입니다.'
        )

    settings = CATEGORY_SETTINGS[category]
    video_id = normalize_text(row['video_id'])
    split = canonical_split(row['split'])
    result_dir = RESULT_ROOT / split / safe_name(video_id)
    body_start_sec = float(row['body_start_sec'])
    body_end_sec = float(row['body_end_sec'])

    if body_end_sec <= body_start_sec:
        raise ValueError(
            f'{video_id}: BODY_END_SEC가 본문 시작시간보다 작거나 같습니다.'
        )

    hash_payload = {
        'queries': queries,
        'body_start_sec': body_start_sec,
        'body_end_sec': body_end_sec,
        'settings': settings,
        'weights': [SALIENCY_WEIGHT, FOREGROUND_WEIGHT],
        'score_aggregation': SCORE_AGGREGATION,
    }
    config_hash = hashlib.sha1(
        json.dumps(hash_payload, ensure_ascii=False, sort_keys=True).encode('utf-8')
    ).hexdigest()[:12]

    return {
        'video_id': video_id,
        'split': split,
        'creator_name': normalize_text(row['creator_name']),
        'title': normalize_text(row['title']),
        'category': category,
        'mode': mode,
        'video_path': Path(row['video_path']),
        'source_video_stem': normalize_text(row['source_video_stem']),
        'body_start_sec': body_start_sec,
        'body_end_sec': body_end_sec,
        'queries': queries,
        'query_hash': config_hash,
        'score_aggregation': SCORE_AGGREGATION,
        'create_preview': CREATE_PREVIEW,
        **settings,
        'result_dir': result_dir,
        'output_video': result_dir / f'{safe_name(video_id)}_{mode}_top4.mp4',
    }


settings_table = pd.DataFrame([
    {
        'category': category,
        'query_count_per_mode': QUERY_COUNT,
        'analysis_clip_sec': settings['analysis_clip_sec'],
        'candidate_top_n': settings['candidate_top_n'],
        'final_top_k': settings['final_top_k'],
        'segment_sec': settings['before_sec'] + settings['after_sec'],
        'top4_preview_sec': settings['final_top_k'] * (
            settings['before_sec'] + settings['after_sec']
        ),
    }
    for category, settings in CATEGORY_SETTINGS.items()
])
display(settings_table)

## 8. 본문 영상과 CLIP Feature 캐시

In [ ]:
import json
import tempfile
from datetime import datetime, timezone


def run_command(command):
    command = [str(value) for value in command]
    print('$', ' '.join(command))
    subprocess.run(command, check=True)


def video_duration(path: Path) -> float:
    result = subprocess.run(
        [
            'ffprobe', '-v', 'error',
            '-show_entries', 'format=duration',
            '-of', 'json', str(path),
        ],
        check=True,
        capture_output=True,
        text=True,
    )
    return float(json.loads(result.stdout)['format']['duration'])


def minmax(values):
    values = np.asarray(values, dtype=np.float32)
    low, high = values.min(), values.max()
    if high - low < 1e-8:
        return np.zeros_like(values)
    return (values - low) / (high - low)


def work_paths(config):
    source = config['video_path']
    stat = source.stat()
    signature = hashlib.sha1(
        json.dumps(
            {
                'path': str(source.resolve()),
                'size': stat.st_size,
                'mtime': stat.st_mtime_ns,
                'body_start': config['body_start_sec'],
                'body_end': config['body_end_sec'],
                'clip_sec': config['analysis_clip_sec'],
            },
            sort_keys=True,
        ).encode('utf-8')
    ).hexdigest()[:12]

    root = WORK_ROOT / f"{safe_name(config['video_id'])}_{signature}"
    feature_dir = root / 'features'
    root.mkdir(parents=True, exist_ok=True)
    feature_dir.mkdir(parents=True, exist_ok=True)

    return {
        'root': root,
        'body': root / 'body_360p.mp4',
        'feature_dir': feature_dir,
        'vid_npz': feature_dir / 'vid.npz',
        'temp_preview': root / f"{config['mode']}_preview.mp4",
    }


def prepare_body(config, paths, rebuild=False):
    source_duration = video_duration(config['video_path'])
    start_sec = max(0.0, config['body_start_sec'])
    end_sec = min(config['body_end_sec'], source_duration)
    duration_sec = end_sec - start_sec

    if duration_sec <= 0:
        raise ValueError(
            f"{config['video_id']}: 유효한 본문 구간이 없습니다. "
            f'start={start_sec}, end={end_sec}, source={source_duration}'
        )

    if rebuild or not paths['body'].exists():
        run_command([
            'ffmpeg', '-y',
            '-ss', f'{start_sec:.3f}',
            '-i', config['video_path'],
            '-t', f'{duration_sec:.3f}',
            '-vf', 'scale=-2:360,fps=30',
            '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '25',
            '-an', '-movflags', '+faststart',
            paths['body'],
        ])
    else:
        print('본문 캐시 재사용:', paths['body'])

    return video_duration(paths['body'])


def extract_video_features(config, paths, rebuild=False):
    if rebuild and paths['vid_npz'].exists():
        paths['vid_npz'].unlink()

    if not paths['vid_npz'].exists():
        print('영상 Feature 추출 시작')
        vid2clip(
            clip_model,
            str(paths['body']),
            str(paths['feature_dir']),
            model_version=CLIP_MODEL_NAME,
            output_feat_size=512,
            clip_len=config['analysis_clip_sec'],
            overwrite=True,
            num_decoding_thread=2,
            half_precision=False,
        )
    else:
        print('영상 Feature 캐시 재사용:', paths['vid_npz'])

    features = np.load(paths['vid_npz'])['features']
    print('Feature shape:', features.shape)
    return features

## 9. Query별 UniVTG 점수 계산·Query 점수 보존

In [ ]:
def load_feature_tensors(feature_dir: Path):
    video = np.load(feature_dir / 'vid.npz')['features'].astype(np.float32)
    text = np.load(feature_dir / 'txt.npz')['features'].astype(np.float32)

    video = torch.from_numpy(l2_normalize_np_array(video))
    text = torch.from_numpy(l2_normalize_np_array(text))

    length = video.shape[0]
    temporal_start = torch.arange(0, length, dtype=torch.float32) / length
    temporal_end = temporal_start + 1.0 / length
    temporal_endpoint = torch.stack([temporal_start, temporal_end], dim=1)
    video = torch.cat([video, temporal_endpoint], dim=1)

    src_vid = video.unsqueeze(0).to(DEVICE)
    src_txt = text.unsqueeze(0).to(DEVICE)
    src_vid_mask = torch.ones(src_vid.shape[:2], device=DEVICE)
    src_txt_mask = torch.ones(src_txt.shape[:2], device=DEVICE)
    return src_vid, src_txt, src_vid_mask, src_txt_mask


@torch.no_grad()
def run_query(query: str, config: dict, paths: dict):
    txt2clip(clip_model, query, str(paths['feature_dir']))
    src_vid, src_txt, src_vid_mask, src_txt_mask = load_feature_tensors(
        paths['feature_dir']
    )

    output = univtg_model(
        src_vid=src_vid,
        src_txt=src_txt,
        src_vid_mask=src_vid_mask,
        src_txt_mask=src_txt_mask,
    )

    saliency = output['saliency_scores'][0].detach().float().cpu().numpy()
    foreground = (
        output['pred_logits'][0]
        .detach().float().cpu().squeeze(-1).numpy()
    )

    clip_sec = config['analysis_clip_sec']
    count = min(len(saliency), len(foreground))
    return pd.DataFrame({
        'clip_index': np.arange(count),
        'start_body_sec': np.arange(count) * clip_sec,
        'end_body_sec': (np.arange(count) + 1) * clip_sec,
        'saliency': saliency[:count],
        'foreground': foreground[:count],
        'query': query,
    })


def aggregate_query_scores(query_scores: np.ndarray, method: str) -> np.ndarray:
    if method == 'max':
        return query_scores.max(axis=0)
    if method == 'mean':
        return query_scores.mean(axis=0)
    if method == 'top2_mean':
        if query_scores.shape[0] < 2:
            return query_scores.mean(axis=0)
        return np.sort(query_scores, axis=0)[-2:].mean(axis=0)
    raise ValueError(f'지원하지 않는 Query 결합 방식: {method}')


def score_all_queries(config, paths):
    query_results = []
    for number, query in enumerate(config['queries'], start=1):
        print(f"Query {number}/{len(config['queries'])}: {query}")
        query_results.append(run_query(query, config, paths))

    common_count = min(len(frame) for frame in query_results)
    query_results = [frame.iloc[:common_count].copy() for frame in query_results]

    saliency_raw = np.vstack([frame['saliency'].to_numpy() for frame in query_results])
    foreground_raw = np.vstack([frame['foreground'].to_numpy() for frame in query_results])
    saliency_norm = np.vstack([minmax(row) for row in saliency_raw])
    foreground_norm = np.vstack([minmax(row) for row in foreground_raw])
    query_scores = (
        SALIENCY_WEIGHT * saliency_norm
        + FOREGROUND_WEIGHT * foreground_norm
    )

    best_query_index_zero = query_scores.argmax(axis=0)
    clip_index = np.arange(query_scores.shape[1])

    result = query_results[0][
        ['clip_index', 'start_body_sec', 'end_body_sec']
    ].copy()
    result['saliency'] = saliency_raw[best_query_index_zero, clip_index]
    result['foreground'] = foreground_raw[best_query_index_zero, clip_index]
    result['combined_score'] = aggregate_query_scores(
        query_scores,
        config['score_aggregation'],
    )
    result['best_query_index'] = best_query_index_zero + 1
    result['best_query'] = [
        config['queries'][index] for index in best_query_index_zero
    ]

    # Top-50 후보에서도 Query별 점수를 직접 비교할 수 있도록 모두 보존합니다.
    for index, query in enumerate(config['queries'], start=1):
        array_index = index - 1
        result[f'query_{index}_saliency_raw'] = saliency_raw[array_index]
        result[f'query_{index}_foreground_raw'] = foreground_raw[array_index]
        result[f'query_{index}_saliency_norm'] = saliency_norm[array_index]
        result[f'query_{index}_foreground_norm'] = foreground_norm[array_index]
        result[f'query_{index}_score'] = query_scores[array_index]

    return result

## 10. Query 기반 Top-50 후보 생성·최종 Top-4 Preview

In [ ]:
def context_window(anchor_start, anchor_end, duration, before, after):
    center = (anchor_start + anchor_end) / 2.0
    segment_length = before + after
    start = center - before
    end = center + after

    if start < 0:
        start = 0.0
        end = min(duration, segment_length)
    if end > duration:
        end = duration
        start = max(0.0, duration - segment_length)
    return start, end


def build_top50_candidates(scores, config, body_duration):
    # Query 결합 점수 기준 순수 Top-50입니다.
    # 이 단계에서는 NMS를 적용하지 않아 Eval에서 K/NMS를 다시 실험할 수 있습니다.
    ranked_clips = (
        scores
        .sort_values(['combined_score', 'clip_index'], ascending=[False, True])
        .head(config['candidate_top_n'])
        .reset_index(drop=True)
    )

    rows = []
    base_columns = [
        'saliency', 'foreground', 'combined_score',
        'best_query_index', 'best_query',
    ]
    query_score_columns = [
        column for column in ranked_clips.columns
        if column.startswith('query_')
    ]

    for candidate_rank, (_, row) in enumerate(ranked_clips.iterrows(), start=1):
        start, end = context_window(
            float(row['start_body_sec']),
            float(row['end_body_sec']),
            body_duration,
            config['before_sec'],
            config['after_sec'],
        )
        candidate = {
            'candidate_rank': candidate_rank,
            'anchor_clip_index': int(row['clip_index']),
            'anchor_start_body_sec': float(row['start_body_sec']),
            'anchor_end_body_sec': float(row['end_body_sec']),
            'anchor_center_body_sec': float(
                (row['start_body_sec'] + row['end_body_sec']) / 2.0
            ),
            'start_body_sec': float(start),
            'end_body_sec': float(end),
            'start_full_sec': float(start + config['body_start_sec']),
            'end_full_sec': float(end + config['body_start_sec']),
            'duration_sec': float(end - start),
        }
        for column in [*base_columns, *query_score_columns]:
            value = row[column]
            if isinstance(value, np.generic):
                value = value.item()
            candidate[column] = value
        rows.append(candidate)

    candidates = pd.DataFrame(rows)
    if candidates.empty:
        raise RuntimeError('Top-50 후보가 생성되지 않았습니다.')

    candidates['is_final_selected'] = False
    candidates['final_rank'] = np.nan
    candidates['preview_order'] = np.nan
    candidates['final_selection_pass'] = ''
    return candidates


def intervals_conflict(start, end, selected_rows, min_gap):
    return any(
        start < old['end_body_sec'] + min_gap
        and end > old['start_body_sec'] - min_gap
        for old in selected_rows
    )


def select_final_topk(candidates, config):
    selected = []
    selected_ids = set()

    def add_pass(min_gap, pass_name, allow_overlap=False):
        for _, row in candidates.sort_values('candidate_rank').iterrows():
            anchor = int(row['anchor_clip_index'])
            if anchor in selected_ids:
                continue
            if not allow_overlap and intervals_conflict(
                float(row['start_body_sec']),
                float(row['end_body_sec']),
                selected,
                min_gap,
            ):
                continue

            item = row.to_dict()
            item['final_selection_pass'] = pass_name
            selected.append(item)
            selected_ids.add(anchor)
            if len(selected) >= config['final_top_k']:
                return

    # 1차: 기존 카테고리별 간격 유지
    add_pass(config['final_min_gap_sec'], 'category_gap')
    # 2차: 추가 간격만 제거하고 구간 중복은 금지
    if len(selected) < config['final_top_k']:
        add_pass(0.0, 'non_overlap')
    # 아주 짧은 영상만 최후 수단으로 중복 허용
    if len(selected) < config['final_top_k']:
        add_pass(0.0, 'overlap_fallback', allow_overlap=True)

    final = pd.DataFrame(selected)
    if final.empty:
        raise RuntimeError('최종 Top-K 장면이 선택되지 않았습니다.')

    final = final.head(config['final_top_k']).reset_index(drop=True)
    final['final_rank'] = np.arange(1, len(final) + 1)
    preview_order = final.sort_values('start_full_sec').reset_index(drop=True)
    preview_order['preview_order'] = np.arange(1, len(preview_order) + 1)

    final_rank_map = final.set_index('anchor_clip_index')['final_rank'].to_dict()
    pass_map = final.set_index('anchor_clip_index')['final_selection_pass'].to_dict()
    preview_map = preview_order.set_index('anchor_clip_index')['preview_order'].to_dict()

    candidates = candidates.copy()
    candidates['is_final_selected'] = candidates['anchor_clip_index'].isin(final_rank_map)
    candidates['final_rank'] = candidates['anchor_clip_index'].map(final_rank_map)
    candidates['preview_order'] = candidates['anchor_clip_index'].map(preview_map)
    candidates['final_selection_pass'] = (
        candidates['anchor_clip_index'].map(pass_map).fillna('')
    )

    if len(candidates) < config['candidate_top_n']:
        print(
            f"주의: 본문 Clip 수가 부족하여 Top-{config['candidate_top_n']} 대신 "
            f'{len(candidates)}개 후보를 저장합니다.'
        )
    if len(final) < config['final_top_k']:
        print(
            f"주의: 최종 Top-{config['final_top_k']} 중 {len(final)}개만 선택되었습니다."
        )
    return candidates, final, preview_order


def render_preview(config, paths, preview_order):
    with tempfile.TemporaryDirectory() as temp_name:
        temp_dir = Path(temp_name)
        parts = []

        for index, row in preview_order.iterrows():
            part = temp_dir / f'part_{index:02d}.mp4'
            run_command([
                'ffmpeg', '-y',
                '-ss', f"{row['start_full_sec']:.3f}",
                '-i', config['video_path'],
                '-t', f"{row['duration_sec']:.3f}",
                '-map', '0:v:0', '-map', '0:a?',
                '-vf', 'scale=-2:720,fps=30',
                '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '22',
                '-c:a', 'aac', '-b:a', '128k', '-ar', '44100', '-ac', '2',
                '-movflags', '+faststart',
                part,
            ])
            parts.append(part)

        concat_file = temp_dir / 'concat.txt'
        concat_file.write_text(
            ''.join(f"file '{part}'\n" for part in parts),
            encoding='utf-8',
        )
        run_command([
            'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
            '-i', concat_file, '-c', 'copy', '-movflags', '+faststart',
            paths['temp_preview'],
        ])

    config['result_dir'].mkdir(parents=True, exist_ok=True)
    shutil.copy2(paths['temp_preview'], config['output_video'])
    return config['output_video']

## 11. Top-50 후보·Top-4·Query Manifest 저장

In [ ]:
# 최종 통합 CSV 경로입니다. 개별 실행 중에는 영상·모드별 파일로 저장하고,
# 배치 종료 후 rebuild_master_csvs()가 한 번에 결합합니다.
SUMMARY_CSV = RESULT_ROOT / 'run_summary.csv'
CANDIDATES_CSV = RESULT_ROOT / 'candidate_segments_top50.csv'
SEGMENTS_CSV = RESULT_ROOT / 'selected_segments.csv'
QUERY_MANIFEST_CSV = RESULT_ROOT / 'query_manifest.csv'


def per_run_paths(config):
    config['result_dir'].mkdir(parents=True, exist_ok=True)
    prefix = f"{safe_name(config['video_id'])}_{config['mode']}"
    return {
        'summary': config['result_dir'] / f'{prefix}_run_summary.csv',
        'candidates': config['result_dir'] / f'{prefix}_candidate_segments_top50.csv',
        'selected': config['result_dir'] / f'{prefix}_selected_segments.csv',
        'query_manifest': config['result_dir'] / f'{prefix}_query_manifest.csv',
        'all_clip_scores': config['result_dir'] / f'{prefix}_all_clip_scores.csv.gz',
    }


def write_csv_atomic(frame: pd.DataFrame, path: Path, *, compression=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(
        temporary,
        index=False,
        encoding='utf-8-sig',
        compression=compression,
    )
    temporary.replace(path)


def completed_with_same_config(config):
    paths = per_run_paths(config)
    if not paths['summary'].exists() or not paths['candidates'].exists():
        return False
    if config['create_preview'] and not config['output_video'].exists():
        return False

    summary = pd.read_csv(paths['summary'], encoding='utf-8-sig', dtype=str)
    required = {'video_id', 'mode', 'query_hash', 'status', 'candidate_count'}
    if summary.empty or not required.issubset(summary.columns):
        return False

    row = summary.iloc[-1]
    if (
        row['video_id'] != config['video_id']
        or row['mode'] != config['mode']
        or row['query_hash'] != config['query_hash']
        or row['status'] != 'success'
    ):
        return False

    expected_count = int(float(row['candidate_count']))
    candidates = pd.read_csv(
        paths['candidates'],
        encoding='utf-8-sig',
        usecols=['video_id', 'mode', 'query_hash'],
        dtype=str,
    )
    actual_count = len(candidates[
        (candidates['video_id'] == config['video_id'])
        & (candidates['mode'] == config['mode'])
        & (candidates['query_hash'] == config['query_hash'])
    ])
    return actual_count == expected_count and actual_count > 0


def save_all_clip_scores(config, scores):
    if not SAVE_ALL_CLIP_SCORES:
        return ''
    paths = per_run_paths(config)
    output = scores.copy()
    output.insert(0, 'video_id', config['video_id'])
    output.insert(1, 'split', config['split'])
    output.insert(2, 'category', config['category'])
    output.insert(3, 'mode', config['mode'])
    output.insert(4, 'query_hash', config['query_hash'])
    write_csv_atomic(output, paths['all_clip_scores'], compression='gzip')
    return str(paths['all_clip_scores'])


def save_success(
    config,
    output_video,
    candidates,
    final,
    preview_order,
    all_clip_score_path='',
):
    paths = per_run_paths(config)
    now = datetime.now(timezone.utc).isoformat()
    segment_length = config['before_sec'] + config['after_sec']
    output_path = str(output_video) if output_video is not None else ''

    summary = pd.DataFrame([{
        'video_id': config['video_id'],
        'split': config['split'],
        'creator_name': config['creator_name'],
        'title': config['title'],
        'category': config['category'],
        'mode': config['mode'],
        'status': 'success',
        'query_hash': config['query_hash'],
        'query_count': len(config['queries']),
        'score_aggregation': config['score_aggregation'],
        'body_start_sec': config['body_start_sec'],
        'body_end_sec': config['body_end_sec'],
        'candidate_top_n': config['candidate_top_n'],
        'candidate_count': len(candidates),
        'final_top_k': config['final_top_k'],
        'final_count': len(final),
        # 이전 Eval 코드와 호환하기 위해 top_k는 최종 Top-K로 유지합니다.
        'top_k': config['final_top_k'],
        'segment_sec': segment_length,
        'preview_sec': round(preview_order['duration_sec'].sum(), 3),
        'preview_enabled': config['create_preview'],
        'output_path': output_path,
        'candidate_path': str(paths['candidates']),
        'selected_path': str(paths['selected']),
        'query_manifest_path': str(paths['query_manifest']),
        'all_clip_score_path': all_clip_score_path,
        'error': '',
        'updated_at_utc': now,
    }])
    write_csv_atomic(summary, paths['summary'])

    query_manifest = pd.DataFrame([
        {
            'video_id': config['video_id'],
            'split': config['split'],
            'creator_name': config['creator_name'],
            'category': config['category'],
            'mode': config['mode'],
            'query_hash': config['query_hash'],
            'query_index': index,
            'query_text': query,
        }
        for index, query in enumerate(config['queries'], start=1)
    ])
    write_csv_atomic(query_manifest, paths['query_manifest'])

    candidate_output = candidates.copy()
    metadata_columns = {
        'video_id': config['video_id'],
        'split': config['split'],
        'creator_name': config['creator_name'],
        'title': config['title'],
        'category': config['category'],
        'mode': config['mode'],
        'query_hash': config['query_hash'],
        'score_aggregation': config['score_aggregation'],
    }
    for position, (column, value) in enumerate(metadata_columns.items()):
        candidate_output.insert(position, column, value)
    write_csv_atomic(candidate_output, paths['candidates'])

    selected_output = final.copy()
    final_preview_map = preview_order.set_index('anchor_clip_index')['preview_order'].to_dict()
    selected_output['preview_order'] = selected_output['anchor_clip_index'].map(final_preview_map)
    for position, (column, value) in enumerate(metadata_columns.items()):
        selected_output.insert(position, column, value)
    selected_output['score_rank'] = selected_output['final_rank']
    write_csv_atomic(selected_output, paths['selected'])


def save_failure(config, error):
    paths = per_run_paths(config)
    for key in ('candidates', 'selected', 'query_manifest'):
        paths[key].unlink(missing_ok=True)

    summary = pd.DataFrame([{
        'video_id': config['video_id'],
        'split': config['split'],
        'creator_name': config['creator_name'],
        'title': config['title'],
        'category': config['category'],
        'mode': config['mode'],
        'status': 'failed',
        'query_hash': config['query_hash'],
        'query_count': len(config['queries']),
        'score_aggregation': config['score_aggregation'],
        'body_start_sec': config['body_start_sec'],
        'body_end_sec': config['body_end_sec'],
        'candidate_top_n': config['candidate_top_n'],
        'candidate_count': 0,
        'final_top_k': config['final_top_k'],
        'final_count': 0,
        'top_k': config['final_top_k'],
        'segment_sec': config['before_sec'] + config['after_sec'],
        'preview_sec': '',
        'preview_enabled': config['create_preview'],
        'output_path': str(config['output_video']),
        'candidate_path': str(paths['candidates']),
        'selected_path': str(paths['selected']),
        'query_manifest_path': str(paths['query_manifest']),
        'all_clip_score_path': '',
        'error': repr(error),
        'updated_at_utc': datetime.now(timezone.utc).isoformat(),
    }])
    write_csv_atomic(summary, paths['summary'])


def combine_per_run_csv(pattern: str, output_path: Path, sort_columns: list[str]):
    files = sorted(RESULT_ROOT.rglob(pattern))
    if not files:
        print(f'결합 대상 없음: {pattern}')
        return pd.DataFrame()
    frames = [
        pd.read_csv(path, encoding='utf-8-sig', low_memory=False)
        for path in files
    ]
    combined = pd.concat(frames, ignore_index=True, sort=False)
    available_sort = [column for column in sort_columns if column in combined.columns]
    if available_sort:
        combined = combined.sort_values(available_sort).reset_index(drop=True)
    write_csv_atomic(combined, output_path)
    print(f'{output_path.name}: {len(combined):,}행 / {len(files):,}개 파일 결합')
    return combined


def rebuild_master_csvs():
    print('\n' + '=' * 80)
    print('개별 결과 → 통합 CSV 재구성')
    print('=' * 80)
    summary = combine_per_run_csv(
        '*_run_summary.csv',
        SUMMARY_CSV,
        ['split', 'category', 'creator_name', 'video_id', 'mode'],
    )
    candidates = combine_per_run_csv(
        '*_candidate_segments_top50.csv',
        CANDIDATES_CSV,
        ['split', 'category', 'creator_name', 'video_id', 'mode', 'candidate_rank'],
    )
    selected = combine_per_run_csv(
        '*_selected_segments.csv',
        SEGMENTS_CSV,
        ['split', 'category', 'creator_name', 'video_id', 'mode', 'score_rank'],
    )
    query_manifest = combine_per_run_csv(
        '*_query_manifest.csv',
        QUERY_MANIFEST_CSV,
        ['split', 'category', 'creator_name', 'video_id', 'mode', 'query_index'],
    )
    return {
        'summary': summary,
        'candidates': candidates,
        'selected': selected,
        'query_manifest': query_manifest,
    }


def run_mode(row, mode, *, overwrite=False, show_video=False):
    config = build_config(row, mode)
    print('\n' + '=' * 72)
    print(config['video_id'], '|', config['split'], '|', config['mode'], '|', config['category'])
    print('본문:', config['body_start_sec'], '~', config['body_end_sec'], '초')
    print('Query:', len(config['queries']), '개')
    print('후보 저장: Top-', config['candidate_top_n'], sep='')
    print(
        '최종 Preview:',
        config['before_sec'] + config['after_sec'],
        '초 ×',
        config['final_top_k'],
    )
    print('저장 폴더:', config['result_dir'])

    if not overwrite and completed_with_same_config(config):
        print('동일 설정 Top-50 결과 재사용')
        if show_video and config['output_video'].exists():
            display(Video(str(config['output_video']), embed=True, width=720))
        return {
            'status': 'skipped',
            'config': config,
            'output_video': config['output_video'],
        }

    paths = work_paths(config)
    try:
        body_duration = prepare_body(config, paths, rebuild=False)
        extract_video_features(config, paths, rebuild=False)
        scores = score_all_queries(config, paths)
        all_clip_score_path = save_all_clip_scores(config, scores)

        candidates = build_top50_candidates(scores, config, body_duration)
        candidates, final, preview_order = select_final_topk(candidates, config)

        output_video = None
        if config['create_preview']:
            output_video = render_preview(config, paths, preview_order)

        save_success(
            config,
            output_video,
            candidates,
            final,
            preview_order,
            all_clip_score_path=all_clip_score_path,
        )

        print('\nTop-50 중 상위 10개')
        display(candidates[[
            'candidate_rank', 'anchor_clip_index',
            'start_full_sec', 'end_full_sec', 'duration_sec',
            'combined_score', 'best_query_index', 'best_query',
            'is_final_selected', 'final_rank',
        ]].head(10))
        print('최종 Top-4')
        display(final[[
            'final_rank', 'candidate_rank', 'start_full_sec', 'end_full_sec',
            'duration_sec', 'combined_score', 'best_query',
            'final_selection_pass',
        ]])

        if show_video and output_video is not None:
            display(Video(str(output_video), embed=True, width=720))

        return {
            'status': 'success',
            'config': config,
            'output_video': output_video,
            'candidates': candidates,
            'final': final,
        }
    except Exception as error:
        save_failure(config, error)
        raise


def run_video_pair(row, *, overwrite=False, show_video=False):
    results = {}
    for mode in ('fixed', 'individual'):
        try:
            results[mode] = run_mode(
                row,
                mode,
                overwrite=overwrite,
                show_video=show_video,
            )
        except Exception as error:
            print(f"{row['video_id']} / {mode} 실패:", repr(error))
            results[mode] = {'status': 'failed', 'error': repr(error)}
    return results

## 12. 영상 한 개 테스트

In [ ]:
# RUN_VIDEO_ID = None   # 예: '-RZQ1mz1tFA'; None이면 첫 번째 실행 가능 영상
# RUN_SPLIT = None      # 예: 'test_a'; None이면 split 제한 없음
# OVERWRITE = False

# single_pool = runnable
# if RUN_SPLIT is not None:
#     split_key = canonical_split(RUN_SPLIT)
#     single_pool = single_pool[single_pool['split'] == split_key]

# if RUN_VIDEO_ID is None:
#     if single_pool.empty:
#         raise RuntimeError('선택 조건에 맞는 영상이 없습니다.')
#     selected_row = single_pool.iloc[0]
# else:
#     key = normalize_text(RUN_VIDEO_ID)
#     matches = single_pool[single_pool['_video_key'] == key]
#     if len(matches) != 1:
#         raise ValueError(f'RUN_VIDEO_ID 매칭 행 수: {len(matches)}')
#     selected_row = matches.iloc[0]

# print('선택 영상:', selected_row['video_id'])
# print('Split:', selected_row['split'])
# print('유튜버:', selected_row['creator_name'])
# print('제목:', selected_row['title'])
# print('원본:', selected_row['video_path'])

# PAIR_RESULT = run_video_pair(
#     selected_row,
#     overwrite=OVERWRITE,
#     show_video=False,
# )

# print('\nFixed:', PAIR_RESULT['fixed'].get('output_video'))
# print('Individual:', PAIR_RESULT['individual'].get('output_video'))
# MASTER_RESULTS = rebuild_master_csvs()
# print('Top-50 CSV:', CANDIDATES_CSV)

## 13. Split 선택형 소규모 또는 전체 배치 실행

In [ ]:

# Final-920 배치 실행
# 처음 테스트하고 싶으면 BATCH_LIMIT_OVERRIDE = 3으로 바꾸세요.
BATCH_LIMIT_OVERRIDE = None
BATCH_SPLITS = None
OVERWRITE = False

batch_rows = runnable.copy()
if BATCH_SPLITS is not None:
    normalized_splits = {canonical_split(value) for value in BATCH_SPLITS}
    batch_rows = batch_rows[batch_rows['split'].isin(normalized_splits)]

batch_rows = batch_rows.sort_values(
    ['split', 'category', 'creator_name', 'video_id']
).reset_index(drop=True)

if BATCH_LIMIT_OVERRIDE is not None:
    batch_rows = batch_rows.head(BATCH_LIMIT_OVERRIDE)
elif FINAL_TARGET_VIDEO_COUNT is not None:
    if len(batch_rows) < FINAL_TARGET_VIDEO_COUNT:
        message = (
            f'실행 가능한 영상 수가 목표 {FINAL_TARGET_VIDEO_COUNT}개보다 적습니다: '
            f'{len(batch_rows)}개'
        )
        if STRICT_TARGET_VIDEO_COUNT:
            raise RuntimeError(message)
        print('[경고]', message)
    elif len(batch_rows) > FINAL_TARGET_VIDEO_COUNT:
        print(
            f'[알림] 실행 가능한 영상 {len(batch_rows)}개 중 '
            f'앞 {FINAL_TARGET_VIDEO_COUNT}개만 Final-920 대상으로 사용합니다.'
        )
        batch_rows = batch_rows.head(FINAL_TARGET_VIDEO_COUNT)

batch_manifest_path = RESULT_ROOT / 'batch_manifest_final920.csv'
batch_rows[[
    'video_id', 'split', 'category', 'creator_name', 'title', 'video_path', 'video_match'
]].to_csv(batch_manifest_path, index=False, encoding='utf-8-sig')

print('배치 영상 수:', len(batch_rows))
print('목표 영상 수:', FINAL_TARGET_VIDEO_COUNT)
print('Batch manifest:', batch_manifest_path)
print('Split 분포:')
display(batch_rows.groupby('split').size().rename('video_count').reset_index())
print('Category 분포:')
display(batch_rows.groupby('category').size().rename('video_count').reset_index())
print('예상 Fixed/Individual 실행 수:', len(batch_rows) * 2)
print('후보 CSV 예상 최대 행:', len(batch_rows) * 2 * 50)
print('Preview 생성:', CREATE_PREVIEW)

for number, (_, row) in enumerate(batch_rows.iterrows(), start=1):
    print('\n' + '#' * 80)
    print(
        f"[{number}/{len(batch_rows)}] "
        f"{row['split']} / {row['video_id']} / {row['creator_name']}"
    )
    print('#' * 80)
    run_video_pair(row, overwrite=OVERWRITE, show_video=False)

MASTER_RESULTS = rebuild_master_csvs()

print('\n배치 완료')
print('결과 폴더:', RESULT_ROOT)
print('Summary:', SUMMARY_CSV)
print('Top-50 후보:', CANDIDATES_CSV)
print('최종 Top-4:', SEGMENTS_CSV)
print('Query Manifest:', QUERY_MANIFEST_CSV)

if SUMMARY_CSV.exists():
    summary = pd.read_csv(SUMMARY_CSV, encoding='utf-8-sig')
    display(summary.tail(len(batch_rows) * 2))

if CANDIDATES_CSV.exists():
    candidates = pd.read_csv(CANDIDATES_CSV, encoding='utf-8-sig', low_memory=False)
    counts = (
        candidates.groupby(['split', 'video_id', 'mode'])
        .size()
        .rename('candidate_count')
        .reset_index()
    )
    display(counts.tail(len(batch_rows) * 2))
